## In-Text Keyword Occurence Search by Topic

Searches all acts for occurences of relevant keywords for a given topic.

Acts and associated keywords are loaded from `topic_data.json`, which was generated from the 24-topic LDA model trained on act-level data.

The code checks `token.lemma_` and `token.text` to catch cases where LatinCy's lemmatization diverges from the form stored in the LDA vocab.

For each match, the code outputs the act ID, surface form, lemma, sentence context, morphological features, and part of speech.

In [2]:
import json
import spacy
from spacy.tokens import DocBin
from collections import defaultdict

nlp = spacy.load("la_core_web_lg")

In [3]:
with open("./Acts/latin_tragedies_acts.json", "r", encoding="utf-8") as f:
    raw_corpus = json.load(f)

In [8]:
#Run only once to serialize + skip on subsequent runs
doc_bin = DocBin()
for entry in raw_corpus:
    doc = nlp(entry["text"])
    doc_bin.add(doc)

doc_bin.to_disk("./Acts/latin_tragedies_acts.spacy")

In [4]:
# Run every time to reconstruct corpus/ids
ids = [entry["id"] for entry in raw_corpus]

In [11]:
with open("./topic_data.json", "r", encoding="utf-8") as f:
    topic_data = json.load(f)

topic_number = 12
target_words = set(topic_data[str(topic_number)]["keywords"]) #use set() for faster lookup
target_doc_ids = set(topic_data[str(topic_number)]["doc_ids"])

# Dictionary to hold lemma occurrences
word_occurences = defaultdict(set)

# Load the DocBin from disk
doc_bin = DocBin().from_disk("./Acts/latin_tragedies_acts.spacy")

docs = list(doc_bin.get_docs(nlp.vocab))

# For each lemma, collect all token occurrences with detailed info            
for doc_id, doc in zip(ids, docs):
    if doc_id not in target_doc_ids:
        continue  # Skip documents not in the target set
    for token in doc:
        if token.lemma_.lower() in target_words or token.text.lower() in target_words:
            matched_word = token.lemma_ if token.lemma_.lower() in target_words else token.text.lower()
            word_occurences[matched_word].add((doc_id, token.text, token.lemma_, token.sent, token.morph, token.pos_))

### Sort by keyword

In [8]:
# Display results by lemma, then by document ID and word
if word_occurences:
    for matched_word, items in word_occurences.items():
        print(f"\n{matched_word}:")
        for doc_id, word, lemma, sentence, morph, pos in sorted(
            items, key=lambda x: (x[0], x[1])
            ):  
            print(f"  Play ID: {doc_id}")
            print(f"    Word: {word}")
            print(f"    Lemma: {lemma}")
            print(f"    Context: {sentence}")
            print(f"    Morph: {morph}")
            print(f"    POS: {pos}")
            print()
else:
        print(f"No occurrences for {', '.join(target_words)}.")


sceptrum:
  Play ID: seneca-agamemnon_act2
    Word: sceptra
    Lemma: sceptrum
    Context: licuit pudicos coniugis quondam toros et sceptra casta vidua tutari fide— periere mores ius decus pietas fides et qui redire cum perit nescit pudor;
    Morph: Case=Acc|Gender=Neut|Number=Plur
    POS: NOUN

  Play ID: seneca-agamemnon_act2
    Word: sceptra
    Lemma: sceptrum
    Context: Pelopia Phrygiae sceptra dum teneant nurus?
    Morph: Case=Acc|Gender=Neut|Number=Plur
    POS: NOUN

  Play ID: seneca-agamemnon_act2
    Word: sceptri
    Lemma: sceptrum
    Context: licet et chorda graviore sones, quale canebas cum Titanas fulmine victos videre dei, vel cum montes montibus altis super impositi struxere gradus trucibus monstris, stetit imposita Pelion Ossa, pinifer ambos pressit Olympus, ades, o magni, soror et coniunx, consors sceptri, regia Iuno:
    Morph: Case=Gen|Gender=Neut|Number=Sing
    POS: NOUN

  Play ID: seneca-hercules-furens_act3
    Word: sceptro
    Lemma: sceptrum
   

### Sort by act id and then keyword

In [12]:
# Display word_occurences sorted by act ID first, then keyword
act_first = defaultdict(lambda: defaultdict(set))

for matched_word, items in word_occurences.items():
    for doc_id, word, lemma, sentence, morph, pos in items:
        act_first[doc_id][matched_word].add((word, lemma, sentence, morph, pos))

if act_first:
    for doc_id in sorted(act_first):
        print(f"\n{doc_id}:")
        for matched_word in sorted(act_first[doc_id]):
            print(f"\n  {matched_word}:")
            for word, lemma, sentence, morph, pos in sorted(
                act_first[doc_id][matched_word], key=lambda x: x[0]
            ):
                print(f"    Word: {word}")
                print(f"    Lemma: {lemma}")
                print(f"    Context: {sentence}")
                print(f"    Morph: {morph}")
                print(f"    POS: {pos}")
                print()
else:
    print(f"No occurrences for {', '.join(target_words)}.")


mussato-ecerinis_act3:

  certus:
    Word: certa
    Lemma: certus
    Context: disposita sidera peragunt cursus vagos dub lege certa.
    Morph: Case=Abl|Gender=Fem|Number=Sing
    POS: ADJ

    Word: certas
    Lemma: certus
    Context: Quae pallet hieme, tempore aestatis viret, certasque certis mensibus fruges alit tellus.
    Morph: Case=Acc|Gender=Fem|Number=Plur
    POS: ADJ

    Word: certis
    Lemma: certus
    Context: Quae pallet hieme, tempore aestatis viret, certasque certis mensibus fruges alit tellus.
    Morph: Case=Abl|Gender=Masc|Number=Plur
    POS: ADJ

    Word: certis
    Lemma: certus
    Context: Terra mare caelum et illa, quae substant eis, gerunt statutas legibus certis vices.
    Morph: Case=Abl|Gender=Fem|Number=Plur
    POS: ADJ

    Word: certo
    Lemma: certus
    Context: Audi negandum, teste nisi certo, novum:
    Morph: Case=Abl|Gender=Neut|Number=Sing
    POS: ADJ


  colo:
    Word: coli
    Lemma: colo
    Context: Iustus hanc coli voluit Deus a

### Look up keywords by specific act id

In [9]:
word_occurences = defaultdict(set)

target_doc = "mussato-ecerinis_act3"  

for doc_id, doc in zip(ids, docs):
    if doc_id != target_doc:
        continue
    for token in doc:
        if token.lemma_.lower() in target_words or token.text.lower() in target_words:
            matched_word = token.lemma_ if token.lemma_.lower() in target_words else token.text.lower()
            word_occurences[matched_word].add((doc_id, token.text, token.lemma_, token.sent, token.morph, token.pos_))


if word_occurences:
    for matched_word, items in word_occurences.items():
        print(f"\n{matched_word}:")
        for doc_id, word, lemma, sentence, morph, pos in sorted(
            items, key=lambda x: (x[0], x[1])
            ):  
            print(f"  Play ID: {doc_id}")
            print(f"    Word: {word}")
            print(f"    Lemma: {lemma}")
            print(f"    Context: {sentence}")
            print(f"    Morph: {morph}")
            print(f"    POS: {pos}")
            print()
else:
    print(f"No occurrences for {', '.join(target_words)}.")            


uis:
  Play ID: mussato-ecerinis_act3
    Word: Vi
    Lemma: uis
    Context: Vi amissa.
    Morph: Case=Abl|Gender=Fem|Number=Sing
    POS: NOUN

  Play ID: mussato-ecerinis_act3
    Word: vi
    Lemma: uis
    Context: Amissa vi?
    Morph: Case=Abl|Gender=Fem|Number=Sing
    POS: NOUN

  Play ID: mussato-ecerinis_act3
    Word: vires
    Lemma: uis
    Context: Fortuna vires ausibus nostris dabit.
    Morph: Case=Acc|Gender=Fem|Number=Plur
    POS: NOUN

  Play ID: mussato-ecerinis_act3
    Word: vires
    Lemma: uis
    Context: O mi frater, o magno sate Plutone, tantis ausibus vires ferat, tellure rupta spiritus nocuos pater nobis faventes commodet;
    Morph: Case=Acc|Gender=Fem|Number=Plur
    POS: NOUN


certus:
  Play ID: mussato-ecerinis_act3
    Word: certa
    Lemma: certus
    Context: disposita sidera peragunt cursus vagos dub lege certa.
    Morph: Case=Abl|Gender=Fem|Number=Sing
    POS: ADJ

  Play ID: mussato-ecerinis_act3
    Word: certas
    Lemma: certus
    Cont